In [1]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score, classification_report

# Import your custom preprocessing module
from preprocessing_22 import load_data_scaled 

# Qiskit imports
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel

In [3]:
print("Loading and scaling data via custom preprocessing module...")
X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test, feature_columns = load_data_scaled()

# --- THE FIX: Align the y indexes with the new X_scaled DataFrames ---
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
# ---------------------------------------------------------------------

# Exact top 10 features identified from Phase 3 SHAP analysis
top_shap_features = [
    'DER_mass_MMC', 'DER_mass_transverse_met_lep', 'DER_mass_vis', 
    'PRI_tau_pt', 'DER_deltar_tau_lep', 'DER_met_phi_centrality', 
    'PRI_jet_leading_eta', 'PRI_lep_eta', 'DER_sum_pt', 'DER_lep_eta_centrality'
]

num_features = len(top_shap_features)
print(f"Filtering dataset to the top {num_features} quantum-ready features...")
X_train_reduced = X_train_scaled[top_shap_features]
X_test_reduced = X_test_scaled[top_shap_features]

# QUANTUM BOTTLENECK: Subset the data
train_subset_size = 10000
test_subset_size = 2000

# Subsetting Train Data
train_indices = X_train_reduced.sample(n=train_subset_size, random_state=42).index
X_train_q = X_train_reduced.loc[train_indices].values
y_train_q = y_train.loc[train_indices].values

# Subsetting Test Data
test_indices = X_test_reduced.sample(n=test_subset_size, random_state=42).index
X_test_q = X_test_reduced.loc[test_indices].values
y_test_q = y_test.loc[test_indices].values

print("Data successfully preprocessed, subsets created, and indexes aligned!")

Loading and scaling data via custom preprocessing module...
Filtering dataset to the top 10 quantum-ready features...
Data successfully preprocessed, subsets created, and indexes aligned!


In [ ]:
print("--- Initializing QSVM Architecture ---")
# Quantum Kernel Mapping
feature_map = ZZFeatureMap(feature_dimension=num_features, reps=2, entanglement='linear')
q_kernel = FidelityQuantumKernel(feature_map=feature_map)

# Use standard scikit-learn SVC, passing the quantum kernel
qsvm = SVC(kernel=q_kernel.evaluate)

print("Fitting QSVM... (This will take a while to evaluate the kernel matrix)")
qsvm.fit(X_train_q, y_train_q)

--- Initializing QSVM Architecture ---
Fitting QSVM... (This will take a while to evaluate the kernel matrix)


C:\Users\nabia\AppData\Local\Temp\ipykernel_23468\4123813571.py:3: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=num_features, reps=2, entanglement='linear')


In [1]:
print("--- QSVM Evaluation ---")
qsvm_pred = qsvm.predict(X_test_q)

# Calculate all required metrics
accuracy = accuracy_score(y_test_q, qsvm_pred)
auc = roc_auc_score(y_test_q, qsvm_pred)
precision = precision_score(y_test_q, qsvm_pred)
recall = recall_score(y_test_q, qsvm_pred)
f1 = f1_score(y_test_q, qsvm_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"AUC:       {auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("\nClassification Report:\n", classification_report(y_test_q, qsvm_pred))

--- QSVM Evaluation ---


NameError: name 'qsvm' is not defined